In [1]:
import cafe
import scanpy as sc

cafe.settings.backend = "python_function"
cafe.logger.setLevel("INFO")

[2025年10月09日 21时37分35秒] INFO                                                                                 
                                          _____     _ _ ______    _       ______            _                      
                                         / ____|   | | |  ____|  | |     |  ____|          | |                     
                                        | |     ___| | | |__ __ _| |_ ___| |__  __  ___ __ | | ___  _ __ ___ _ __  
                                        | |    / _ \ | |  __/ _` | __/ _ \  __| \ \/ / '_ \| |/ _ \| '__/ _ \ '__| 
                                        | |___|  __/ | | | | (_| | ||  __/ |____ >  <| |_) | | (_) | | |  __/ |    
                                         \_____\___|_|_|_|  \__,_|\__\___|______/_/\_\ .__/|_|\___/|_|  \___|_|    
                                                                                     | |                           
                                                                              

数据

In [2]:
fadata = cafe.data.read_scvelo_pancrease(n_obs=500)
fadata

AttributeError: module 'cafe.data' has no attribute 'read_scvelo_pancrease'

In [ ]:
import pandas as pd

# 手动添加参考里程碑
milestone_network = pd.DataFrame(
    data=[
        ["Ductal", "Ngn3 low EP"],
        ["Ngn3 low EP", "Ngn3 high EP"],
        ["Ngn3 high EP", "Pre-endocrine"],
        ["Pre-endocrine", "Alpha"],
        ["Pre-endocrine", "Beta"],
        ["Pre-endocrine", "Delta"],
        ["Pre-endocrine", "Epsilon"],
        ],
    columns=["from", "to"]
)

fadata.add_trajectory_mannually(milestone_network)

In [ ]:
cluster_key = "milestone_color"
fadata.group_onto_nearest_milestones(cluster_key=cluster_key) # new cluster color

basis = "umap"
cluster_key_list = ["milestone", cluster_key]
cafe.plot.plot_graph(fadata, color=cluster_key_list)
cafe.plot.plot_trajectory(fadata, basis=basis, color=cluster_key_list)


In [ ]:
cafe.metric.metrics

,metric_id,plotmath,latex,html,long_name,category,type,perfect,worst,symmetric
0,correlation,cor[dist],\mathit{cor}_{\textrm{dist}},cor<sub>dist</sub>,Geodesic distance correlation,cell positions,specific,1,0.0,True
1,rf_mse,MSE[rf],\mathit{MSE}_{\textit{rf}},MSE<sub>rf</sub>,Random Forest MSE,neighbourhood,specific,0,0.3,False
2,rf_nmse,NMSE[rf],\mathit{NMSE}_{\textit{rf}},NMSE<sub>rf</sub>,Random Forest Normalised MSE,neighbourhood,specific,1,0.0,False
3,rf_rsq,R[rf]^2,R^{2}_{rf},R<sup>2</sup><sub>rf</sub>,Random Forest R²,neighbourhood,specific,1,0.0,False
4,lm_nmse,NMSE[lm],\mathit{NMSE}_{\textit{lm}},NMSE<sub>lm</sub>,Linear regression Normalised MSE,neighbourhood,specific,1,0.0,False
5,lm_mse,MSE[lm],\mathit{MSE}_{\textit{lm}},MSE<sub>lm</sub>,Linear regression MSE,neighbourhood,specific,0,0.3,False
6,lm_rsq,R[lm]^2,R^{2}_{lm},R<sup>2</sup><sub>lm</sub>,Linear regression R²,neighbourhood,specific,1,0.0,False
7,edge_flip,edgeflip,\textrm{edgeflip},edgeflip,Edge flip,topology,specific,1,0.0,True
8,him,HIM,\textrm{HIM},HIM,Hamming-Ipsen-Mikhailov similarity,topology,specific,1,0.0,True
9,isomorphic,isomorphic,\textrm{isomorphic},Isomorphic,isomorphic,topology,specific,1,0.0,True


In [ ]:
# TODO: PAGA need to be optimized, disconnected graph
prior_information = {
    "start_id": fadata.obs.index[0],
    "groups_id": fadata.obs[cluster_key].tolist()
}
parameters = {"filter_features": False, "connectivity_cutoff": 0.3}
fadata.add_prior_information(**prior_information)  # add prior information to fadata


#method_name_list = ["paga", "comp1", "angle", "state_comp", "cluster_mst", "projection_mst", "graph_mst", "scvelo"]
method_name_list = ["comp1", "angle", "state_comp","paga","cluster_mst","projection_mst","graph_mst"]


for method_name in method_name_list:
        method = cafe.method.FateMethod(method_name=method_name)
        method.infer_trajectory(fadata)
        cafe.plot.plot_trajectory(fadata, basis="umap", color=cluster_key_list)
        

查看模型名称

In [ ]:
parsed_model_name_list = fadata.get_all_model_name() # 解析后的模型名称
model_name_list = fadata.get_all_model_name(parse=False)
parsed_model_name_list, model_name_list

构造新fadata结构（给代码的补丁）

In [ ]:
fadata.X

In [ ]:
    new_fadata = cafe.data.FateAnnData(
        X=fadata.X,
        obs=fadata.obs,
        uns=fadata.uns
    )
    new_fadata

In [ ]:
fadata.trajectory_history_dict

输出模型对应的MilestoneWrapper,并给新的new_fadata添加信息

In [ ]:
for i in model_name_list:
    print(f"{i} : {fadata.trajectory_history_dict[i]['milestone_wrapper']}")
    new_fadata.model_name = i
    new_fadata.milestone_wrapper = fadata.trajectory_history_dict[i]['milestone_wrapper']

检查new_fadata结构

In [ ]:
new_fadata

In [ ]:
new_fadata.uns["cfe"]['trajectory_history_dict']

查看metric信息

In [ ]:
cafe.metric.metrics

In [ ]:
implemented = [
    "correlation",
    "rf_mse", "rf_rsq", "rf_nmse",
    "lm_mse", "lm_rsq", "lm_nmse",
    "edge_flip", "him",
    "featureimp_cor", "featureimp_wcor",
    "featureimp_ks", "featureimp_wilcox",
    "F1_branches", "F1_milestones"
]
implemented

In [ ]:
models = list(new_fadata.uns["cfe"]["trajectory_history_dict"].keys())
# 4. 确保每个模型都有 waypoint_wrapper
for m in set(models) | {"ref"}:
    new_fadata.model_name = m
    if not new_fadata.is_wrapped_with_waypoints:
        new_fadata.add_waypoints()
# 4. 针对每个方法，调用 calculate_metrics 并收集结果
records = {}
for model in models:
    # 跳过参考自身（ref vs ref）若不想算，可以加 if model=="ref": continue
    res = cafe.metric.calculate_metrics(
        new_fadata,
        now_model=model,
        ref_model="ref",
        simplify=True,
        metrics=implemented
    )
    # 把可能缺失的指标填成 NaN
    for m in implemented:
        res.setdefault(m, float("nan"))
    records[model] = res

# 5. 构造成 DataFrame
df = pd.DataFrame.from_dict(records, orient="index", columns=implemented)

# 6. （可选）把行索引改成更可读的名字，或保存到文件
df.index.name = "method"
df

尝试运行caculate_metric得到字典

In [ ]:
metric_dict=cafe.metric.

## 指标

In [ ]:
parsed_model_name_list = fadata.get_all_model_name() # 解析后的模型名称
model_name_list = fadata.get_all_model_name(parse=False)
parsed_model_name_list, model_name_list

In [ ]:
fadata.uns["cfe"]

In [ ]:
metrics = cafe.metric.metrics
metric_id_list = metrics[metrics["category"] == "topology"]["metric_id"].tolist()
metric_id_list

In [ ]:
import pandas as pd


model_metric_dict = {}

for model_name in model_name_list:
    if "angle-python_function" not in model_name: # angle模型
        model_metric = cafe.metric.calculate_metrics(
            fadata,
            metrics=metric_id_list,
            now_model=model_name,
            ref_model="ref"
        )
        model_metric_dict[model_name] = model_metric

df = pd.DataFrame(model_metric_dict).T
df